In [1]:


import mlflow

with mlflow.start_run():
    mlflow.log_param("param1", 15)
    mlflow.log_metric("metric1", 0.89)

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [4]:
df.dropna(inplace=True)

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df = df[~(df['clean_comment'].str.strip() == '')]

In [7]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [8]:
# Ensure necessary NLTK data is downloaded
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

In [10]:
# Apply the preprocessing function to the 'clean_comment' column
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)

In [11]:
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [12]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=10000)  # Bag of Words model with a limit of 1000 features

In [14]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']  # Assuming 'sentiment' is the target variable (0 or 1 for binary classification)

In [ ]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
X.shape

(36793, 10000)

In [ ]:
y

,category
0,1
1,1
2,-1
3,0
4,1
...,...
37244,0
37245,1
37246,0
37247,1


In [ ]:
y.shape

(36793,)

In [ ]:
# Step 2: Set up the MLflow tracking server
##mlflow.set_tracking_uri("http://ec2-54-196-109-131.compute-1.amazonaws.com:5000/")

In [15]:
# ========================= TRAIN TEST SPLIT =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [16]:
# Set or create an experiment# ========================= MLFLOW SETUP =========================
mlflow.set_tracking_uri("file:./mlruns")

mlflow.set_experiment("Baseline RandomForest Experiment")

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/574253305047331370'), creation_time=1778700657693, experiment_id='574253305047331370', last_update_time=1778700657693, lifecycle_stage='active', name='Baseline RandomForest Experiment', tags={}, trace_location=None, workspace='default'>

In [18]:


# ========================= SPLIT DATA =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ========================= START RUN =========================
with mlflow.start_run() as run:

    # ========================= TAGS =========================
    mlflow.set_tag(
        "mlflow.runName",
        "RandomForest_Baseline_TrainTestSplit"
    )

    mlflow.set_tag(
        "experiment_type",
        "baseline"
    )

    mlflow.set_tag(
        "model_type",
        "RandomForestClassifier"
    )

    mlflow.set_tag(
        "description",
        "Baseline RandomForest model using CountVectorizer"
    )

    # ========================= PARAMETERS =========================
    mlflow.log_param(
        "vectorizer_type",
        "CountVectorizer"
    )

    mlflow.log_param(
        "vectorizer_max_features",
        vectorizer.max_features
    )

    n_estimators = 200
    max_depth = 15

    mlflow.log_param(
        "n_estimators",
        n_estimators
    )

    mlflow.log_param(
        "max_depth",
        max_depth
    )

    # ========================= MODEL =========================
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # ========================= TRAIN =========================
    model.fit(X_train, y_train)

    # ========================= PREDICT =========================
    y_pred = model.predict(X_test)

    # ========================= ACCURACY =========================
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    # ========================= CLASSIFICATION REPORT =========================
    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():

        if isinstance(metrics, dict):

            for metric, value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    conf_matrix = confusion_matrix(
        y_test,
        y_pred
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    ax.set_title(
        "Confusion Matrix"
    )

    mlflow.log_figure(
        fig,
        "confusion_matrix.png"
    )

    plt.close(fig)

    # =========================================================
    # CLASS DISTRIBUTION
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.countplot(
        x=y,
        ax=ax
    )

    ax.set_title(
        "Dataset Class Distribution"
    )

    ax.set_xlabel("Classes")
    ax.set_ylabel("Count")

    mlflow.log_figure(
        fig,
        "class_distribution.png"
    )

    plt.close(fig)

    # =========================================================
    # FEATURE IMPORTANCE
    # =========================================================
    feature_importance = model.feature_importances_

    top_n = 20

    indices = feature_importance.argsort()[-top_n:]

    feature_names = vectorizer.get_feature_names_out()

    top_features = feature_names[indices]

    top_importance = feature_importance[indices]

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        top_features,
        top_importance
    )

    ax.set_title(
        "Top 20 Important Features"
    )

    ax.set_xlabel("Importance")

    mlflow.log_figure(
        fig,
        "feature_importance.png"
    )

    plt.close(fig)

    # =========================================================
    # PREDICTION DISTRIBUTION
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.countplot(
        x=y_pred,
        ax=ax
    )

    ax.set_title(
        "Prediction Distribution"
    )

    ax.set_xlabel("Predicted Classes")
    ax.set_ylabel("Count")

    mlflow.log_figure(
        fig,
        "prediction_distribution.png"
    )

    plt.close(fig)

    # =========================================================
    # SAVE MODEL
    # =========================================================
    mlflow.sklearn.log_model(
        model,
        "random_forest_model"
    )

    # =========================================================
    # SAVE DATASET
    # =========================================================
    df.to_csv(
        "dataset.csv",
        index=False
    )

    mlflow.log_artifact(
        "dataset.csv"
    )

# ========================= FINAL OUTPUT =========================
print(f"Accuracy: {accuracy}")

2026/05/16 20:54:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 20:54:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.6479141187661367


In [23]:

# ========================= START MLFLOW RUN =========================
with mlflow.start_run() as run:

    # ========================= TAGS =========================
    mlflow.set_tag(
        "mlflow.runName",
        "RandomForest_Baseline_TrainTestSplit"
    )

    mlflow.set_tag(
        "experiment_type",
        "baseline"
    )

    mlflow.set_tag(
        "model_type",
        "RandomForestClassifier"
    )

    mlflow.set_tag(
        "description",
        "Baseline RF model using CountVectorizer"
    )

    # ========================= PARAMETERS =========================
    mlflow.log_param(
        "vectorizer_type",
        "CountVectorizer"
    )

    mlflow.log_param(
        "vectorizer_max_features",
        vectorizer.max_features
    )

    n_estimators = 200
    max_depth = 15

    mlflow.log_param(
        "n_estimators",
        n_estimators
    )

    mlflow.log_param(
        "max_depth",
        max_depth
    )

    # ========================= MODEL =========================
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # ========================= TRAIN =========================
    model.fit(X_train, y_train)

    # ========================= PREDICT =========================
    y_pred = model.predict(X_test)

    # ========================= ACCURACY =========================
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    # ========================= CLASSIFICATION REPORT =========================
    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():

        if isinstance(metrics, dict):

            for metric, value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    conf_matrix = confusion_matrix(
        y_test,
        y_pred
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")

    plt.tight_layout()

    mlflow.log_figure(
        fig,
        "confusion_matrix.png"
    )

    plt.close(fig)

    # =========================================================
    # CLASS DISTRIBUTION
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.countplot(
        x=y,
        ax=ax
    )

    ax.set_title("Dataset Class Distribution")
    ax.set_xlabel("Classes")
    ax.set_ylabel("Count")

    plt.xticks(rotation=45)

    plt.tight_layout()

    mlflow.log_figure(
        fig,
        "class_distribution.png"
    )

    plt.close(fig)

    # =========================================================
    # FEATURE IMPORTANCE
    # =========================================================
    feature_importance = model.feature_importances_

    top_n = 20

    indices = feature_importance.argsort()[-top_n:]

    feature_names = vectorizer.get_feature_names_out()

    top_features = feature_names[indices]

    top_importance = feature_importance[indices]

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        top_features,
        top_importance
    )

    ax.set_title("Top 20 Important Features")

    ax.set_xlabel("Importance")

    plt.tight_layout()

    mlflow.log_figure(
        fig,
        "feature_importance.png"
    )

    plt.close(fig)

    # =========================================================
    # PREDICTION DISTRIBUTION
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.countplot(
        x=y_pred,
        ax=ax
    )

    ax.set_title("Prediction Distribution")

    ax.set_xlabel("Predicted Classes")

    ax.set_ylabel("Count")

    plt.xticks(rotation=45)

    plt.tight_layout()

    mlflow.log_figure(
        fig,
        "prediction_distribution.png"
    )

    plt.close(fig)

    # =========================================================
    # SAVE MODEL
    # =========================================================
    mlflow.sklearn.log_model(
        model,
        "random_forest_model"
    )

    # =========================================================
    # SAVE DATASET
    # =========================================================
    df.to_csv(
        "dataset.csv",
        index=False
    )

    mlflow.log_artifact(
        "dataset.csv"
    )

# ========================= FINAL OUTPUT =========================
print(f"Accuracy: {accuracy}")

2026/05/14 01:16:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 01:16:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.6479141187661367


In [19]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          -1       1.00      0.01      0.02      1650
           0       0.68      0.81      0.74      2555
           1       0.62      0.85      0.72      3154

    accuracy                           0.65      7359
   macro avg       0.77      0.56      0.49      7359
weighted avg       0.73      0.65      0.57      7359



In [21]:
df.to_csv('reddit_preprocessing.csv', index=False)

In [22]:
pd.read_csv('reddit_preprocessing.csv').head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
